# 수정본의 실행·해석 범위

이 노트북은 2026-10-07 코드 수정본입니다. 저장된 출력과 과거 성능 숫자는 제거했습니다. 실제 Excel 전체 처리, 모델 학습·탐색과 성능 재평가는 이번 수정에서 실행하지 않았습니다. 노트북 폴더를 작업 디렉터리로 사용하고 원자료와 `ml_validation.py`, 프로젝트 의존성을 준비한 뒤 위에서 아래로 실행해야 합니다.

`수익률`은 기존 Excel 값을 신뢰하지 않고 매수가·매도가·구매주식수·투자원금으로 다시 계산합니다. 수수료·세금·배당을 포함하지 않은 계산입니다. 이전에 열람한 2021년 이후 자료는 **과거 자료 재평가**이며, 새로 봉인된 최종 테스트가 아닙니다. 전년도 재무제표라는 이유만으로 매수 당시 공개된 정보라고 보장할 수 없으므로 공시 시점 자료의 연결은 별도 과제입니다. 입력 계약과 한계는 `NOTEBOOK-REPAIR-NOTES.md`를 확인하세요.

GridSearch와 RandomizedSearch 셀은 실제 실행 시 많은 트리를 학습합니다. 기본 탐색은 각각 243회+재학습, 300회+재학습으로 가벼운 문법 확인과 다릅니다. 이번 수정에서는 실행하지 않았습니다.

현재 보관된 `ALL_profit_final(개).xlsx`의 13,979행 중 3행은 매도가가 없습니다(Excel 행 942, 952, 2536). 이 파일을 그대로 읽으면 `recalculate_returns`가 의도적으로 오류를 냅니다. 결측을 0으로 바꾸거나 행을 자동 삭제해 학습을 진행하지 않습니다. 원가격·보유기간 자료를 확인해 처리 근거를 정하기 전에는 전체 학습을 재개할 수 없습니다. 초기 회계 진단 비교에서 유효 13,976행의 포지션 손익 부호 141개가 양수에서 비양수로 바뀌었습니다. 이 수치는 새 가격상승 정답의 변경 개수와 다릅니다. 이는 모델 성능 검증이 아닙니다. 근거: `saved-label-audit.json`.

## 분류 목표를 가격 상승으로 분리했습니다

현재 목표는 **수정주가 기준 가격 상승 여부**입니다. `add_price_direction_targets(recalculate_returns(frame))`로 회계 열을 유지한 뒤 `가격변화율 = 100 × (매도가 / 매수가 − 1)`, `가격상승여부 = int(매도가 > 매수가)`를 추가합니다. 모델의 정답 y는 `가격상승여부`이고, `수익률_양음`은 배분된 원금·정수 주식 수에 따른 회계 손익 부호로 보존합니다.

원자료의 0주 행 25개 중 가격이 상승한 8행은 포지션 이익 0이지만 가격상승 정답은 1입니다(`verification/zero-share-audit.json`). 수정주가는 당시 체결 가능한 실제 호가와 같다고 보장되지 않습니다. 당시 가격 없이 수정주가로 정수 주식 수를 계산한 백테스트의 현실성, 원자료의 주식분할·배당 등 보정 정의는 아직 확인하지 않았습니다. 따라서 새 목표는 기업 자체의 성장·투자 수익의 증명이 아니라 저장된 수정주가의 기간 변화입니다. 기존 포지션 이익 분류의 성능 숫자를 새 목표의 성능으로 재사용하지 않습니다.


In [ ]:
from sklearn.tree import plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import precision_recall_curve

import numpy as np
import pandas as pd

import KSE_가치팩터_functions as kf

import matplotlib.pyplot as plt
from matplotlib import font_manager, rc
import seaborn as sns
from sklearn.metrics import make_scorer
from ml_validation import recalculate_returns, temporal_split, expanding_year_folds, add_price_direction_targets

In [ ]:
sns.set_theme()

FONT_NAME = font_manager.FontProperties(fname="c:/Windows/Fonts/malgun.ttf").get_name()
rc('font', family=FONT_NAME)

In [ ]:
# 엑셀 파일 경로 (파일 경로는 실제 경로로 수정해야 합니다)
file_path = 'KSE_FIN_DATA_2023.xlsx'

# 전체 데이터셋 로드 (파일 경로와 시트 이름을 실제 데이터에 맞게 수정하세요)
df_close = pd.read_excel(file_path, sheet_name='수정주가')

# 날짜 열을 datetime 형태로 변환
df_close['Symbol Name'] = pd.to_datetime(df_close['Symbol Name'])

close_1 = df_close.copy()

# 첫 번째 열을 인덱스로 설정
close_1.set_index(close_1.columns[0], inplace=True)

# 인덱스를 날짜 형식으로 변환
close_1.index = pd.to_datetime(close_1.index)

In [ ]:
### ALL모델링

In [ ]:
# 회계 손익을 재계산한 뒤, 별도 분류 목표인 수정주가 상승 여부를 추가합니다.
excel_file_path = 'ALL_profit_final(개).xlsx'
excel_data = add_price_direction_targets(recalculate_returns(pd.read_excel(excel_file_path)))
FEATURES = ['1/PER', '1/PBR', '1/PSR', '1/PCR', 'ALL', 'ALL Rank',
            '1/PER Rank', '1/PBR Rank', '1/PSR Rank', '1/PCR Rank']
train_data, validation_data, test_data = temporal_split(excel_data)
X_train = train_data[FEATURES]
y_train = train_data['가격상승여부']
# 이 시점에는 test_data에 예측하지 않습니다.
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, y_train)
# 같은 연도 기업들이 양쪽에 흩어지지 않도록 연도별로 분리합니다.
cv_folds = expanding_year_folds(train_data, n_splits=3)
f1_scorer = make_scorer(f1_score, zero_division=0)

In [ ]:
### 파라미터 튜닝

In [ ]:
# 랜덤 포레스트 파라미터 그리드 설정
param_grid = {
    'n_estimators': [100, 200, 300],  # 트리의 개수
    'max_features': ['sqrt'], # 각 트리에서 고려할 최대 feature 수
    'max_depth': [10, 20, 30],        # 트리의 최대 깊이
    'min_samples_split': [2, 5, 10],  # 내부 노드를 분할하는데 필요한 최소 샘플 수
    'min_samples_leaf': [1, 2, 4]     # 리프 노드에 있어야 할 최소 샘플 수
}

rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(estimator = rf, param_grid = param_grid, cv = cv_folds, scoring = f1_scorer, error_score = "raise", n_jobs = -1, verbose = 2)
grid_search.fit(X_train, y_train)

# 최적의 파라미터 출력
print(grid_search.best_params_)

In [ ]:
# 랜덤 서치 파라미터 분포 설정
param_dist = {
    'n_estimators': randint(100, 500),   # 트리의 개수
    'max_features': ['sqrt'],    # 각 트리에서 고려할 최대 feature 수
    'max_depth': randint(10, 50),        # 트리의 최대 깊이
    'min_samples_split': randint(2, 11), # 내부 노드를 분할하는데 필요한 최소 샘플 수
    'min_samples_leaf': randint(1, 5)    # 리프 노드에 있어야 할 최소 샘플 수
}

rf = RandomForestClassifier(random_state=42)
random_search = RandomizedSearchCV(estimator = rf, param_distributions = param_dist, n_iter = 100, cv = cv_folds, scoring = f1_scorer, error_score = "raise", verbose=2, random_state=42, n_jobs = -1)
random_search.fit(X_train, y_train)

# 최적의 파라미터 출력
print(random_search.best_params_)

In [ ]:
X_val = validation_data[FEATURES]
y_val = validation_data['가격상승여부']

In [ ]:
# refit=True 기본값으로 전체 학습 구간에 재학습된 실제 탐색 결과입니다.
optimized_rf_grid = grid_search.best_estimator_
optimized_rf_random = random_search.best_estimator_
predictions_grid = optimized_rf_grid.predict(X_val)
predictions_random = optimized_rf_random.predict(X_val)

In [ ]:
# 그리드 서치 모델 평가
accuracy_grid = accuracy_score(y_val, predictions_grid)
precision_grid = precision_score(y_val, predictions_grid, zero_division=0)
recall_grid = recall_score(y_val, predictions_grid, zero_division=0)
f1_grid = f1_score(y_val, predictions_grid, zero_division=0)

# 랜덤 서치 모델 평가
accuracy_random = accuracy_score(y_val, predictions_random)
precision_random = precision_score(y_val, predictions_random, zero_division=0)
recall_random = recall_score(y_val, predictions_random, zero_division=0)
f1_random = f1_score(y_val, predictions_random, zero_division=0)

# 결과 출력
print("그리드 서치 모델 평가:")
print(f"정확도: {accuracy_grid}, 정밀도: {precision_grid}, 재현율: {recall_grid}, F1 점수: {f1_grid}")

print("\n랜덤 서치 모델 평가:")
print(f"정확도: {accuracy_random}, 정밀도: {precision_random}, 재현율: {recall_random}, F1 점수: {f1_random}")
# 미리 정한 선택 기준: 검증 F1, 동률이면 grid. 테스트 결과로 선택하지 않습니다.
selected_search_name = 'grid' if f1_grid >= f1_random else 'random'
optimized_model = optimized_rf_grid if selected_search_name == 'grid' else optimized_rf_random
print(f'검증 F1로 선택한 모델: {selected_search_name}')

과거 정확도와 모델 우열 해석은 현재 수정본의 성능이 아닙니다. 정답 계산, 검증 방법, 탐색 결과 연결이 바뀌었으므로 숫자를 재사용하지 않습니다. 위 표는 다시 실행했을 때의 **2017~2020 검증 결과**이며, 탐색 내부 점수나 2021년 이후 평가 점수와 구분해야 합니다.

In [ ]:
# 시각화

In [ ]:
# 특성 중요도 추출
feature_importances = optimized_rf_random.feature_importances_

# 특성명 추출
feature_names = X_train.columns

# 특성 중요도를 DataFrame으로 변환
importances = pd.DataFrame({'feature': feature_names, 'importance': feature_importances})

# 중요도에 따라 내림차순 정렬
importances = importances.sort_values('importance', ascending=False)

# 시각화
plt.figure(figsize=(10,6))
plt.barh(importances['feature'], importances['importance'], color='skyblue')
plt.xlabel('중요도')
plt.ylabel('특성')
plt.title('특성 중요도 in Random Forest Model')
plt.gca().invert_yaxis()  # 높은 중요도가 위로 오도록 y축 반전
plt.show()

In [ ]:
# 혼동 행렬 계산
cm = confusion_matrix(y_val, predictions_random)

# 혼동 행렬 시각화
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('Actual Values')
plt.xlabel('Predicted Values')
plt.show()

In [ ]:
# ROC 곡선을 위한 값 계산
fpr, tpr, thresholds = roc_curve(y_val, optimized_rf_random.predict_proba(X_val)[:,1])
roc_auc = auc(fpr, tpr)

# ROC 곡선 시각화
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.show()

In [ ]:
# 정밀도-재현율 곡선을 위한 값 계산
precision, recall, _ = precision_recall_curve(y_val, optimized_rf_random.predict_proba(X_val)[:,1])

# 정밀도-재현율 곡선 시각화
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='blue', lw=2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.show()

종목 목록은 모델 선택 후 아래 과거 평가 셀에서 생성합니다. 가격상승 분류값이 1인 기업 중 `ALL` 값 순서로 최대 20개를 고르며, 상승 확률 순위가 아닙니다.

In [ ]:
# 튜닝하지 않은 기준모델도 같은 검증 구간으로 비교합니다.
original_predictions_validation = clf.predict(X_val)
print(f'기준모델 검증 정확도: {accuracy_score(y_val, original_predictions_validation)}')
print(confusion_matrix(y_val, original_predictions_validation, labels=[0, 1]))

아래는 이미 과거에 살펴본 2021년 이후 자료의 재평가입니다. 새로운 최종 테스트라고 부르지 않습니다. 선택 기준은 앞선 검증 F1로 고정하며, 이 결과를 보고 모델을 다시 고르면 별도의 새 평가 구간이 필요합니다.

In [ ]:
# optimized_model은 검증 F1로 선택한 search.best_estimator_입니다.
X_test = test_data[FEATURES]
y_test = test_data['가격상승여부']
test_predictions = optimized_model.predict(X_test)
print('과거 자료 재평가 정확도:', accuracy_score(y_test, test_predictions))
print('과거 자료 재평가 F1:', f1_score(y_test, test_predictions, zero_division=0))
print('과거 자료 재평가 혼동행렬:', confusion_matrix(y_test, test_predictions, labels=[0, 1]))
test_results = test_data.assign(예측_가격상승여부=test_predictions)
# 가격 상승으로 예측한 기업 중 ALL 값 순서입니다. 확률 순위/수익 보장이 아닙니다.
top_20_all_year = (test_results.loc[test_results['예측_가격상승여부'] == 1]
                   .sort_values(['연도', 'ALL'], ascending=[True, False])
                   .groupby('연도', group_keys=False).head(20).reset_index(drop=True))
top_20_all_year[['연도', '기업명'] + FEATURES + ['가격상승여부', '예측_가격상승여부']]

In [ ]:
output_filename = "top_20_all_year_randomforest.xlsx"
top_20_all_year.to_excel(output_filename, index=False)

print(f"Data saved to {output_filename}")